# 📊 Embedding Model Comparison

**Choose the right embedding model for your use case**

---

## 📋 Overview

**What you'll learn:**
- Popular embedding models
- Speed vs accuracy tradeoffs
- Benchmarking models
- Domain-specific models
- Model selection guide

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time
from typing import List, Dict
import pandas as pd

print("✅ Setup complete")

## 🎯 Popular Embedding Models

### Model Landscape:

| Model | Dimensions | Speed | Quality | Best For |
|-------|------------|-------|---------|----------|
| **all-MiniLM-L6-v2** | 384 | ⚡⚡⚡ | Good | General, fast |
| **all-mpnet-base-v2** | 768 | ⚡⚡ | Better | Quality matters |
| **multi-qa-mpnet** | 768 | ⚡⚡ | Better | Q&A |
| **e5-large-v2** | 1024 | ⚡ | Best | Research |
| **gte-large** | 1024 | ⚡ | Best | Latest SOTA |
| **OpenAI text-embedding-3** | 1536/3072 | ⚡⚡ | Excellent | Paid API |

### Tradeoffs:
- 📏 **Dimensions**: Higher = more info, slower
- ⚡ **Speed**: Smaller models = faster
- 🎯 **Quality**: Larger models = better accuracy
- 💰 **Cost**: Local = free, API = paid

## ⚡ Speed Benchmark

In [ ]:
def benchmark_speed(model_name: str, texts: List[str], n_runs: int = 3) -> Dict:
    """Benchmark encoding speed."""
    
    print(f"\n⏱️  Testing {model_name}...")
    
    # Load model
    load_start = time.time()
    model = SentenceTransformer(model_name)
    load_time = time.time() - load_start
    
    # Warmup
    _ = model.encode(["warmup"])
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        start = time.time()
        embeddings = model.encode(texts, show_progress_bar=False)
        times.append(time.time() - start)
    
    avg_time = np.mean(times)
    time_per_text = avg_time / len(texts)
    
    return {
        'model': model_name,
        'dimensions': embeddings.shape[1],
        'load_time': load_time,
        'total_time': avg_time,
        'time_per_text': time_per_text,
        'texts_per_sec': 1 / time_per_text
    }

# Test texts
test_texts = [
    "Machine learning models can predict outcomes",
    "Natural language processing helps understand text",
    "Deep learning uses neural networks",
    "Python is great for data science",
    "Cloud computing provides scalable resources",
] * 20  # 100 texts total

# Models to compare
models_to_test = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
]

print("🏃 Running speed benchmarks...")
print(f"Texts: {len(test_texts)}")

results = []
for model_name in models_to_test:
    result = benchmark_speed(model_name, test_texts)
    results.append(result)

# Display results
print("\n📊 Speed Benchmark Results:\n")
df = pd.DataFrame(results)
print(df[['model', 'dimensions', 'total_time', 'texts_per_sec']].to_string(index=False))

fastest = df.loc[df['texts_per_sec'].idxmax()]
print(f"\n⚡ Fastest: {fastest['model']} ({fastest['texts_per_sec']:.1f} texts/sec)")

## 🎯 Quality Benchmark

In [ ]:
def benchmark_quality(model_name: str, test_cases: List[Dict]) -> Dict:
    """
    Benchmark semantic similarity quality.
    
    Args:
        test_cases: [{'text1': ..., 'text2': ..., 'expected_similar': bool}]
    """
    
    model = SentenceTransformer(model_name)
    
    correct = 0
    similarities = []
    
    for case in test_cases:
        emb1 = model.encode([case['text1']])[0]
        emb2 = model.encode([case['text2']])[0]
        
        # Cosine similarity
        similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
        similarities.append(similarity)
        
        # Check if correct (threshold = 0.5)
        predicted_similar = similarity > 0.5
        if predicted_similar == case['expected_similar']:
            correct += 1
    
    accuracy = correct / len(test_cases)
    
    return {
        'model': model_name,
        'accuracy': accuracy,
        'correct': correct,
        'total': len(test_cases),
        'avg_similarity': np.mean(similarities)
    }

# Create test cases
quality_test_cases = [
    # Should be similar
    {"text1": "I love programming", "text2": "I enjoy coding", "expected_similar": True},
    {"text1": "Machine learning", "text2": "Artificial intelligence", "expected_similar": True},
    {"text1": "Buy groceries", "text2": "Shop for food", "expected_similar": True},
    {"text1": "Cat", "text2": "Feline", "expected_similar": True},
    {"text1": "Happy", "text2": "Joyful", "expected_similar": True},
    
    # Should NOT be similar
    {"text1": "I love programming", "text2": "The weather is nice", "expected_similar": False},
    {"text1": "Machine learning", "text2": "Cooking pasta", "expected_similar": False},
    {"text1": "Cat", "text2": "Computer", "expected_similar": False},
    {"text1": "Happy", "text2": "Sad", "expected_similar": False},
    {"text1": "Python", "text2": "Java", "expected_similar": False},  # Tricky!
]

print("🎯 Running quality benchmarks...\n")

quality_results = []
for model_name in models_to_test:
    print(f"Testing {model_name}...")
    result = benchmark_quality(model_name, quality_test_cases)
    quality_results.append(result)

print("\n📊 Quality Benchmark Results:\n")
df_quality = pd.DataFrame(quality_results)
print(df_quality.to_string(index=False))

best = df_quality.loc[df_quality['accuracy'].idxmax()]
print(f"\n🏆 Best quality: {best['model']} ({best['accuracy']*100:.1f}% accuracy)")

## 📏 Dimension Comparison

In [ ]:
# Compare embedding dimensions
models_dims = [
    'all-MiniLM-L6-v2',    # 384
    'all-mpnet-base-v2',   # 768
]

print("📏 Embedding Dimensions Comparison\n")

text = "This is a sample sentence for embedding"

for model_name in models_dims:
    model = SentenceTransformer(model_name)
    embedding = model.encode([text])[0]
    
    print(f"{model_name}:")
    print(f"  Dimensions: {len(embedding)}")
    print(f"  Size in memory: {embedding.nbytes / 1024:.2f} KB")
    print(f"  First 5 values: {embedding[:5]}")
    print()

print("💡 Higher dimensions = more info but slower + more storage")

## 🎨 Domain-Specific Models

In [ ]:
# Domain-specific model guide
domain_models = {
    "General Purpose": [
        {"name": "all-MiniLM-L6-v2", "dims": 384, "use": "Fast, general"},
        {"name": "all-mpnet-base-v2", "dims": 768, "use": "Better quality"},
    ],
    "Question Answering": [
        {"name": "multi-qa-mpnet-base-dot-v1", "dims": 768, "use": "Q&A retrieval"},
        {"name": "multi-qa-MiniLM-L6-cos-v1", "dims": 384, "use": "Fast Q&A"},
    ],
    "Code Search": [
        {"name": "flax-sentence-embeddings/st-codesearch-distilroberta-base", "dims": 768, "use": "Code"},
    ],
    "Scientific": [
        {"name": "allenai-specter", "dims": 768, "use": "Scientific papers"},
    ],
}

print("🎨 Domain-Specific Model Guide\n")
print("="*70)

for domain, models in domain_models.items():
    print(f"\n{domain}:")
    for model in models:
        print(f"  • {model['name']}")
        print(f"    Dimensions: {model['dims']}, Use: {model['use']}")

## 🔬 Advanced: Model Selection Helper

In [ ]:
class ModelSelector:
    """Help choose the right embedding model."""
    
    @staticmethod
    def recommend(use_case: str, priority: str = "balanced") -> Dict:
        """
        Recommend model based on use case and priority.
        
        Args:
            use_case: 'general', 'qa', 'code', 'scientific'
            priority: 'speed', 'quality', 'balanced'
        """
        
        recommendations = {
            'general': {
                'speed': 'all-MiniLM-L6-v2',
                'quality': 'all-mpnet-base-v2',
                'balanced': 'all-MiniLM-L6-v2'
            },
            'qa': {
                'speed': 'multi-qa-MiniLM-L6-cos-v1',
                'quality': 'multi-qa-mpnet-base-dot-v1',
                'balanced': 'multi-qa-MiniLM-L6-cos-v1'
            },
            'code': {
                'speed': 'all-MiniLM-L6-v2',
                'quality': 'flax-sentence-embeddings/st-codesearch-distilroberta-base',
                'balanced': 'all-MiniLM-L6-v2'
            },
        }
        
        model = recommendations.get(use_case, {}).get(priority, 'all-MiniLM-L6-v2')
        
        return {
            'recommended_model': model,
            'use_case': use_case,
            'priority': priority,
            'reason': f"Optimized for {use_case} with {priority} priority"
        }

# Test selector
selector = ModelSelector()

print("🔬 Model Selection Examples\n")
print("="*60)

scenarios = [
    ('general', 'speed'),
    ('general', 'quality'),
    ('qa', 'balanced'),
]

for use_case, priority in scenarios:
    rec = selector.recommend(use_case, priority)
    print(f"\nUse case: {use_case}, Priority: {priority}")
    print(f"  Recommended: {rec['recommended_model']}")
    print(f"  Reason: {rec['reason']}")

## ✅ Summary

### Quick Selection Guide:

**Need fast + good enough?**
```python
✅ all-MiniLM-L6-v2
   - 384 dimensions
   - ~10x faster than large models
   - Good for most use cases
```

**Need best quality?**
```python
✅ all-mpnet-base-v2
   - 768 dimensions
   - Better accuracy
   - 2x slower
```

**Need Q&A specific?**
```python
✅ multi-qa-mpnet-base-dot-v1
   - Trained on Q&A pairs
   - Better for retrieval
```

### Model Comparison:

| Metric | MiniLM | MPNet | E5-large |
|--------|--------|-------|----------|
| Dimensions | 384 | 768 | 1024 |
| Speed | ⚡⚡⚡ | ⚡⚡ | ⚡ |
| Quality | Good | Better | Best |
| Storage/embedding | 1.5 KB | 3 KB | 4 KB |
| Best for | Production | Balanced | Research |

### Benchmarks (MTEB Leaderboard):

```
Model Performance on MTEB:

all-MiniLM-L6-v2:     58.8% avg
all-mpnet-base-v2:    63.3% avg
gte-large:            65.4% avg
e5-large-v2:          64.5% avg
OpenAI ada-002:       60.0% avg
```

### Decision Tree:

```
Do you need real-time search?
├─ Yes → all-MiniLM-L6-v2
└─ No
   ├─ Is accuracy critical?
   │  ├─ Yes → all-mpnet-base-v2
   │  └─ No → all-MiniLM-L6-v2
   └─ Special domain?
      ├─ Code → codesearch model
      ├─ Q&A → multi-qa model
      └─ Scientific → specter
```

### Storage Costs:

For 1 million documents:
```python
MiniLM (384):  1.5 GB
MPNet (768):   3.0 GB
E5 (1024):     4.0 GB
```

### Best Practices:

1. **Start with MiniLM**
   - Good baseline
   - Fast and cheap
   - Upgrade if needed

2. **Benchmark on your data**
   - Generic benchmarks ≠ your performance
   - Test with real queries
   - Measure recall@k

3. **Consider domain**
   - Domain-specific models can be 10-20% better
   - Worth it for specialized use cases

4. **Monitor costs**
   - Larger embeddings = more storage
   - More dimensions = slower search
   - Balance quality vs cost

### Production Recommendations:

**Startup/MVP:**
- Use: all-MiniLM-L6-v2
- Why: Fast, cheap, good enough

**Scale (< 10M docs):**
- Use: all-mpnet-base-v2
- Why: Better quality, manageable cost

**Enterprise:**
- Use: Custom fine-tuned model
- Why: Best performance on your data

### Next: `06_fine_tuning/01_when_to_finetune.ipynb`